In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import joblib
import os

# caricamento dati
data = pd.read_csv("/content/drive/MyDrive/prototipo2/tickets_sintetici.csv")
data["title"] = data["title"].fillna("")
data["title"] = data["title"].str.lower()
data["body"] = data["body"].fillna("")
data["body"] = data["body"].str.lower()
data["text"] = data["title"] + " " + data["body"]

In [ ]:
# feature e target
category_map = {
    "Amministrazione": 0,
    "Tecnico": 1,
    "Commerciale": 2
}

y = data["category"].map(category_map).values

#Perché migliora
# ngram_range=(1,2) --> cattura frasi tipo
#   "password reset"
#   "errore login"
#   "fattura pagamento"
# min_df=2 --> elimina parole rare (rumore)
#max_df=0.9 --> elimina parole troppo frequenti (tipo stopwords)
vectorizer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9
)
X = vectorizer.fit_transform(data["text"]).toarray()

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [ ]:
# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    data["text"], y, test_size=0.2, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(max_features=2000)
X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()
# conversione a tensor
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

In [ ]:
#modello di rete neurale
class TicketClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        # Vantaggi
        # apprende pattern più complessi
        # migliora classificazione del testo
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# instanzia il modello:
#   X.shape[1] corrisponde al numero di ticket
#   3 è il numero di classi target
model_cat = TicketClassifier(X_train.shape[1], 3)
# definisce la loss function per la classificazione multiclasse
criterion = nn.CrossEntropyLoss()
# l'ottimizzatore usa Adam, definendo di andare a ad aggiornare model_cat
#   con un learning rate di 0.001 più stabile di 0.01
optimizer = torch.optim.Adam(model_cat.parameters(), lr=0.001)

In [ ]:
#ciclo di addestramento
for epoch in range(21):
    # pulisce i gradienti calcolati all'epoca precedente
    optimizer.zero_grad()
    # forward pass (predizione)
    outputs = model_cat(X_train)
    # calcolo della loss
    loss = criterion(outputs, y_train)
    # back-propagation
    loss.backward()
    # aggiornamento dei pesi
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.0963
Epoch 5, Loss: 1.0555
Epoch 10, Loss: 1.0038
Epoch 15, Loss: 0.9370
Epoch 20, Loss: 0.8559


In [ ]:
with torch.no_grad():
    outputs = model_cat(X_test)
    preds = torch.argmax(outputs, dim=1)

from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, preds))

Accuracy: 0.97


In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, preds))

[[35  0  3]
 [ 0 35  0]
 [ 0  0 27]]


In [ ]:
from sklearn.metrics import classification_report

classificazione = classification_report(y_test, preds)
print(classificazione)

              precision    recall  f1-score   support

           0       1.00      0.92      0.96        38
           1       1.00      1.00      1.00        35
           2       0.90      1.00      0.95        27

    accuracy                           0.97       100
   macro avg       0.97      0.97      0.97       100
weighted avg       0.97      0.97      0.97       100



In [ ]:
MODEL_DIR = "/content/drive/MyDrive/prototipo2/models"
os.makedirs(MODEL_DIR, exist_ok=True)

torch.save(
    model_cat.state_dict(),
    MODEL_DIR + "/category_model.pt"
)

joblib.dump(
    vectorizer,
    MODEL_DIR + "/tfidf_vectorizer.joblib"
)

print("✅ category_model.pt e TF-IDF salvati")

✅ category_model.pt e TF-IDF salvati
